# VLA 推理与部署教程

本教程将带你了解如何：
- 加载训练好的模型
- 进行推理
- 优化推理速度
- 部署到生产环境

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
import time
import sys
sys.path.insert(0, '/root/.openclaw/workspace/vla-training')

print("✅ 环境准备完成！")

## 1. 加载模型

In [ ]:
# 模拟加载模型
class SimpleVLAModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(100, 256),
            nn.ReLU(),
            nn.Linear(256, 128)
        )
        self.head = nn.Linear(128, 70)
    
    def forward(self, x):
        features = self.encoder(x)
        return self.head(features)

# 创建并加载模型
model = SimpleVLAModel()

# 模拟加载检查点
# checkpoint = torch.load('checkpoints/best_model.pt')
# model.load_state_dict(checkpoint['model_state_dict'])

print("✅ 模型加载完成！")
print(f"模型参数量: {sum(p.numel() for p in model.parameters()):,}")

## 2. 推理模式

In [ ]:
def inference(model, input_data, device='cpu'):
    """
    单次推理
    
    Args:
        model: VLA 模型
        input_data: 输入数据
        device: 运行设备
    
    Returns:
        actions: 预测的动作
        inference_time: 推理时间 (ms)
    """
    model.eval()
    model.to(device)
    
    with torch.no_grad():
        input_tensor = torch.tensor(input_data, dtype=torch.float32).to(device)
        
        # 计时
        start_time = time.time()
        
        # 推理
        output = model(input_tensor)
        
        # 结束计时
        inference_time = (time.time() - start_time) * 1000
        
        actions = output.cpu().numpy()
    
    return actions, inference_time

# 测试推理
input_data = np.random.randn(1, 100)
actions, inference_time = inference(model, input_data)

print(f"预测动作形状: {actions.shape}")
print(f"推理时间: {inference_time:.2f} ms")
print(f"动作范围: [{actions.min():.3f}, {actions.max():.3f}]")

## 3. 批量推理

In [ ]:
def batch_inference(model, inputs, batch_size=32, device='cpu'):
    """批量推理"""
    model.eval()
    model.to(device)
    
    results = []
    total_time = 0
    
    with torch.no_grad():
        for i in range(0, len(inputs), batch_size):
            batch = inputs[i:i+batch_size]
            batch_tensor = torch.tensor(batch, dtype=torch.float32).to(device)
            
            start_time = time.time()
            output = model(batch_tensor)
            total_time += (time.time() - start_time) * 1000
            
            results.append(output.cpu().numpy())
    
    return np.concatenate(results), total_time

# 测试批量推理
batch_inputs = np.random.randn(100, 100)
batch_actions, total_time = batch_inference(model, batch_inputs, batch_size=10)

print(f"批量大小: 100")
print(f"总推理时间: {total_time:.2f} ms")
print(f"平均每条: {total_time/100:.2f} ms")
print(f"吞吐量: {100/(total_time/1000):.1f} samples/sec")

## 4. 推理优化

### 4.1 模型量化

In [ ]:
# 动态量化
quantized_model = torch.quantization.quantize_dynamic(
    model,
    {nn.Linear},
    dtype=torch.qint8
)

# 对比推理速度
input_data = np.random.randn(1, 100)

_, time_fp32 = inference(model, input_data)
_, time_int8 = inference(quantized_model, input_data)

print(f"FP32 推理时间: {time_fp32:.2f} ms")
print(f"INT8 推理时间: {time_int8:.2f} ms")
print(f"加速比: {time_fp32/time_int8:.2f}x")

### 4.2 推理缓存

In [ ]:
class CachedVLAModel(nn.Module):
    """带缓存的 VLA 模型"""
    
    def __init__(self, model):
        super().__init__()
        self.model = model
        self.cache = {}
    
    def forward(self, x, use_cache=True):
        if use_cache:
            # 简单的缓存机制
            x_hash = hash(x.data.tobytes())
            if x_hash in self.cache:
                return self.cache[x_hash]
            output = self.model(x)
            self.cache[x_hash] = output
            return output
        return self.model(x)

print("✅ 缓存机制示例完成！")

### 4.3 ONNX 导出

In [ ]:
print("""
ONNX 导出示例:

```python
# 导出到 ONNX
dummy_input = torch.randn(1, 100)
torch.onnx.export(
    model,
    dummy_input,
    'vla_model.onnx',
    input_names=['input'],
    output_names=['actions'],
    dynamic_axes={
        'input': {0: 'batch_size'},
        'actions': {0: 'batch_size'}
    }
)

# 使用 ONNX Runtime 推理
import onnxruntime as ort
session = ort.InferenceSession('vla_model.onnx')
output = session.run(None, {'input': input_data})
```
""")

## 5. 实时控制循环

In [ ]:
def realtime_control_loop(model, camera, robot, frequency=10):
    """
    实时控制循环
    
    Args:
        model: VLA 模型
        camera: 摄像头对象
        robot: 机器人对象
        frequency: 控制频率 (Hz)
    """
    import time
    
    model.eval()
    
    print(f"开始实时控制，频率: {frequency} Hz")
    
    try:
        while True:
            loop_start = time.time()
            
            # 1. 获取图像
            # image = camera.get_image()
            
            # 2. 预处理
            # input_tensor = preprocess(image)
            
            # 3. 推理
            with torch.no_grad():
                # actions = model(input_tensor)
                pass
            
            # 4. 执行动作
            # robot.execute(actions[0])
            
            # 5. 控制频率
            elapsed = time.time() - loop_start
            sleep_time = max(0, 1/frequency - elapsed)
            time.sleep(sleep_time)
            
    except KeyboardInterrupt:
        print("停止控制循环")

print("✅ 实时控制循环示例完成！")

## 6. 性能基准测试

In [ ]:
def benchmark(model, input_shape=(1, 100), num_runs=100):
    """基准测试"""
    model.eval()
    
    # 预热
    dummy_input = torch.randn(*input_shape)
    for _ in range(10):
        with torch.no_grad():
            _ = model(dummy_input)
    
    # 正式测试
    times = []
    for _ in range(num_runs):
        input_tensor = torch.randn(*input_shape)
        
        start = time.time()
        with torch.no_grad():
            _ = model(input_tensor)
        elapsed = (time.time() - start) * 1000
        times.append(elapsed)
    
    return {
        'mean': np.mean(times),
        'std': np.std(times),
        'min': np.min(times),
        'max': np.max(times),
        'p50': np.percentile(times, 50),
        'p95': np.percentile(times, 95),
        'p99': np.percentile(times, 99)
    }

# 运行基准测试
print("运行基准测试...")
results = benchmark(model, num_runs=100)

print(f"\n基准测试结果:")
print(f"  平均延迟: {results['mean']:.2f} ms")
print(f"  标准差: {results['std']:.2f} ms")
print(f"  最小延迟: {results['min']:.2f} ms")
print(f"  最大延迟: {results['max']:.2f} ms")
print(f"  P50: {results['p50']:.2f} ms")
print(f"  P95: {results['p95']:.2f} ms")
print(f"  P99: {results['p99']:.2f} ms")
print(f"  吞吐量: {1000/results['mean']:.1f} FPS")

## 7. 部署方案

In [ ]:
print("""
## 部署方案对比

| 方案 | 延迟 | 吞吐量 | 适用场景 |
|------|------|--------|----------|
| PyTorch (CPU) | 高 | 低 | 开发测试 |
| PyTorch (GPU) | 中 | 中 | 研究实验 |
| ONNX Runtime | 低 | 高 | 生产部署 |
| TensorRT | 极低 | 极高 | 边缘部署 |
| TorchScript | 低 | 高 | 服务部署 |

## 推荐部署流程

1. **开发阶段**: PyTorch (GPU)
2. **测试阶段**: ONNX Runtime
3. **生产部署**: TensorRT / ONNX Runtime
4. **边缘设备**: TensorRT (Jetson)
""")

## 8. 总结

In [ ]:
print("""
🎉 推理与部署教程完成！

关键要点:
1. 推理时使用 torch.no_grad() 节省内存
2. 模型量化可大幅提升推理速度
3. ONNX/TensorRT 适合生产部署
4. 实时控制需要控制循环频率
5. 基准测试帮助选择最优方案

下一步:
- 尝试导出 ONNX 模型
- 在实际机器人上部署
- 优化推理性能
""")